In [1]:
import os

# bloquear TensorFlow completamente
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [2]:
!pip uninstall -y tensorflow tensorflow-estimator keras
!pip install -q transformers datasets evaluate sentencepiece protobuf==3.20.3

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
google-cloud-firestore 2.27.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigtable 2.38.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.21.0 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
goo

In [1]:
!pip install evaluate bert-score
!pip install sacrebleu
!pip install unbabel-comet

!pip install --upgrade protobuf

  Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl (295 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.1
    Uninstalling protobuf-7.35.1:
      Successfully uninstalled protobuf-7.35.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


  Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl (327 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 7.35.1 which is incompatible.
google-cloud-aiplatform 1.157.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires p

In [2]:
#Cargar Datos

import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [3]:
#Limpieza

import re

def limpiar_texto(texto):
    texto = re.sub(r"[^\w\s-]", "", texto)
    return texto

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

In [4]:
#Silabificador

VOCALES = "aeiou"
CONSONANTES_COMPUESTAS = ["ch", "sh", "ts"]
SUFIJOS = ["klu", "ru", "lu", "chri", "xlu"]

def dividir_palabra(palabra):
    silabas = []
    actual = ""
    i = 0

    while i < len(palabra):
        if i + 1 < len(palabra):
            par = palabra[i:i+2]
            if par in CONSONANTES_COMPUESTAS:
                actual += par
                i += 2
                continue

        letra = palabra[i]
        actual += letra

        if letra in VOCALES:
            silabas.append(actual)
            actual = ""

        i += 1

    if actual:
        silabas.append(actual)

    return silabas


def silabificar_yine(texto):
    palabras = texto.lower().split()
    resultado = []

    for palabra in palabras:
        if len(palabra) <= 4:
            resultado.append(palabra)
            continue

        partes = palabra.split("-")

        for parte in partes:
            sufijo_encontrado = False

            for sufijo in SUFIJOS:
                if parte.endswith(sufijo) and len(parte) > len(sufijo):
                    raiz = parte[:-len(sufijo)]
                    resultado.extend(dividir_palabra(raiz))
                    resultado.append(sufijo)
                    sufijo_encontrado = True
                    break

            if not sufijo_encontrado:
                resultado.extend(dividir_palabra(parte))

    return resultado

In [5]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificar_yine(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
most_common_subwords = subword_counter.most_common(50)
print(most_common_subwords)

# Guardar los nuevos tokens para agregarlos al tokenizador
nuevos_tokens = [token for token, count in most_common_subwords]


SUBWORDS MÁS FRECUENTES:
[('wa', 183), ('ka', 172), ('ta', 160), ('ne', 138), ('ga', 138), ('gi', 123), ('pa', 89), ('ru', 81), ('na', 72), ('chi', 62), ('nu', 56), ('tka', 51), ('ya', 48), ('ge', 43), ('tu', 39), ('pi', 38), ('nka', 37), ('su', 30), ('ko', 29), ('wane', 28), ('ra', 28), ('po', 26), ('ni', 25), ('kle', 25), ('lu', 23), ('no', 23), ('wu', 23), ('yo', 23), ('tlo', 22), ('chri', 22), ('te', 21), ('we', 21), ('mama', 19), ('wiwi', 19), ('tya', 19), ('pga', 19), ('ma', 18), ('papa', 17), ('klu', 16), ('lo', 15), ('gita', 15), ('tna', 15), ('je', 15), ('tje', 15), ('ri', 14), ('tma', 14), ('tni', 14), ('pe', 14), ('t', 13), ('nro', 13)]


In [6]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
# Prepend prompt BEFORE dataset creation (mT5 specific)
df_model["source"] = df_model["source"].apply(
    lambda x: "translate Yine to Spanish: " + x
)
dataset = Dataset.from_pandas(df_model)
print(dataset)



Dataset({
    features: ['source', 'target'],
    num_rows: 372
})


In [7]:
## Tokenizer mT5

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

# =========================
# AGREGAR TOKENS AL TOKENIZER
# =========================
if 'nuevos_tokens' in globals():
    tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
    print("Nuevos tokens reales a agregar:", len(tokens_filtrados))
    tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
    print("Tokens agregados:", tokens_agregados)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Nuevos tokens reales a agregar: 7
Tokens agregados: 7


In [8]:
##Función de tokenización para usar en todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = [
        (l if l != tokenizer.pad_token_id else -100)
        for l in labels["input_ids"]
    ]

    return model_inputs

In [9]:
##Traer el modelo

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Embedding(250107, 512)

In [10]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

#Aplicamos la funcion pta tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])

tokenized_dataset.set_format("torch")

Map:   0%|          | 0/372 [00:00<?, ? examples/s]

In [11]:
##Creación del trainning

from transformers import TrainingArguments
from transformers import Trainer

# ==============================================================================
# PARÁMETROS FASE 1 (ENTRENAMIENTO INICIAL - mT5 para Yine):
# NOTA: El corpus de Yine tiene solo ~372 oraciones. Para que mT5 converja con
# un dataset tan pequeño, necesitamos MAXIMIZAR el número de pasos de optimización:
#
# - per_device_train_batch_size=1: Con batch=1 tenemos 372 pasos/época (vs 24 con batch=4+grad_accum=4).
# - num_train_epochs=30: 30 épocas × 372 pasos = 11,160 pasos totales de optimización.
# - learning_rate=5e-4: Tasa alta para forzar la adaptación desde span corruption hacia traducción.
# - warmup_steps=50: Solo ~0.4% del total de pasos para no desperdiciar épocas en calentamiento.
# - weight_decay=0.01: Regularización ligera.
# - logging_steps=200: Para monitorear la curva de loss durante el entrenamiento.
# ==============================================================================
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=30,
    learning_rate=5e-4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [12]:
def traducir(texto):
    texto_con_prompt = "translate Yine to Spanish: " + texto
    inputs = tokenizer(texto_con_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=50,
        num_beams=4,           #  beam search
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
## Entrenar el modelo

trainer.train()

Step,Training Loss
200,12.900200
400,4.898000
600,3.702400
800,3.327600
1000,2.947100
1200,2.699600
1400,2.554000
1600,2.171300
1800,2.145400
2000,1.843800


TrainOutput(global_step=11160, training_loss=1.1286095202182784, metrics={'train_runtime': 439.0642, 'train_samples_per_second': 25.418, 'train_steps_per_second': 25.418, 'total_flos': 737594848051200.0, 'train_loss': 1.1286095202182784, 'epoch': 30.0})

In [14]:

##Guardar 1era version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4/tokenizer.json')

In [15]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO Y TOKENIZER (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 y tokenizer desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo y tokenizer desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 y tokenizer desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4


The tokenizer you are loading from '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [16]:
# Prompt ya prependeado en la celda de creacion del dataset
print("Prompt ya configurado.")


Prompt ya configurado.


In [17]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
# Prepend prompt BEFORE dataset creation (mT5 specific)
df_model["source"] = df_model["source"].apply(
    lambda x: "translate Yine to Spanish: " + x
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 372
})


In [18]:
# ==============================================================================
# DECLARACIÓN Y PREPROCESAMIENTO DE VAL_DATASET (YINE mT5)
# ==============================================================================
import pandas as pd
from datasets import Dataset

# 1. Cargar el conjunto de validación
path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/val.csv"
df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

# 2. Aplicar la función de limpieza
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

# 3. Formatear y renombrar columnas para Hugging Face
df_val_model = df_val[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)

# 4. Agregar el prompt de traducción para Yine (específico de mT5)
df_val_model["source"] = df_val_model["source"].apply(
    lambda x: "translate Yine to Spanish: " + x
)

# 5. Declarar la variable val_dataset
val_dataset = Dataset.from_pandas(df_val_model)
print("val_dataset cargado con éxito:", val_dataset)


val_dataset cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 46
})


In [19]:
##Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [20]:
#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: katu ganunrotanu
REAL: con quién va a casarse
PRED: vamos a ver al ciempiés
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: mi hermanito come un plátano
-----
INPUT: gi piklu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no tiene miedo
-----
INPUT: wuya gi patewatanutka
REAL: anda no tengas más vergüenza
PRED: vamos a ver la flor
-----
INPUT: gi ge ponro peta
REAL: no has visto al ciempiés
PRED: no ves al ciempiés
-----


In [21]:
##BLEU

import evaluate
bleu = evaluate.load("bleu")

bleu_result = bleu.compute(predictions=preds, references=refs)
print("BLEU:", bleu_result)

BLEU: {'bleu': 0.11198698254413354, 'precisions': [0.3682008368200837, 0.17616580310880828, 0.08163265306122448, 0.0297029702970297], 'brevity_penalty': 1.0, 'length_ratio': 1.0391304347826087, 'translation_length': 239, 'reference_length': 230}


In [24]:
## CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)
print("ChrF:", chrf_result)

ChrF: {'score': 34.340835480783646, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [25]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)
print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.6217322122791539


In [26]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: {'meteor': 0.29017293865048505}


In [27]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Promedio): 0.793684923130533
BERTScore Recall (Promedio): 0.7917543856993966
BERTScore F1 (Promedio): 0.792520637097566


In [28]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL (FASE 2 - mT5 Yine):
# 1. learning_rate=1e-4: Se reduce significativamente para refinamiento fino.
# 2. num_train_epochs=15: Épocas adicionales para refinamiento.
# 3. warmup_steps=30: Calentamiento mínimo.
# 4. Se mantiene batch_size=1 para maximizar pasos de optimización.
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=15,
    learning_rate=1e-4,
    warmup_steps=30,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

In [29]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [30]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [31]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) y tokenizer para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) y tokenizer para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) y tokenizer para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4


The tokenizer you are loading from '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Embedding(250107, 512)

In [32]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [33]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
200,0.090500
400,0.086300
600,0.084400
800,0.087200
1000,0.056600
1200,0.072900
1400,0.073300
1600,0.055100
1800,0.042300
2000,0.042200


TrainOutput(global_step=5580, training_loss=0.044946276372478854, metrics={'train_runtime': 235.0559, 'train_samples_per_second': 23.739, 'train_steps_per_second': 23.739, 'total_flos': 368797424025600.0, 'train_loss': 0.044946276372478854, 'epoch': 15.0})

In [34]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_yine_v4_retrained/tokenizer.json')

In [35]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: katu ganunrotanu
REAL: con quién va a casarse
PRED: vamos a decir
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: mi hermanito come plátanos
-----
INPUT: gi piklu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no es ciego el niño
-----
INPUT: wuya gi patewatanutka
REAL: anda no tengas más vergüenza
PRED: vamos a ver la flor
-----
INPUT: gi ge ponro peta
REAL: no has visto al ciempiés
PRED: no ves al ciempiés
-----


In [36]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [37]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.13175746360087928,
 'precisions': [0.351931330472103,
  0.17647058823529413,
  0.09219858156028368,
  0.05263157894736842],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0130434782608695,
 'translation_length': 233,
 'reference_length': 230}

In [38]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 35.078361551599265, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [39]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.5967790685270143


In [40]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


METEOR (Validación - Reentrenado): {'meteor': 0.27177893313298446}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [41]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


BERTScore Precision (Validación - Reentrenado - Promedio): 0.7835954013078109
BERTScore Recall (Validación - Reentrenado - Promedio): 0.7800782789354739
BERTScore F1 (Validación - Reentrenado - Promedio): 0.7814253659352012


In [42]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [43]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [44]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 47
})


In [45]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [46]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.12579703922303612,
 'precisions': [0.3559322033898305,
  0.21164021164021163,
  0.11267605633802817,
  0.07368421052631578],
 'brevity_penalty': 0.7954768289198056,
 'length_ratio': 0.8137931034482758,
 'translation_length': 236,
 'reference_length': 290}

In [47]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 29.7426212611481, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [48]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.5970627513337643


In [49]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


METEOR: {'meteor': 0.31915496857009273}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [50]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


BERTScore Precision (Promedio): 0.7927867021966488
BERTScore Recall (Promedio): 0.7862355430075463
BERTScore F1 (Promedio): 0.7892113391389238
